In [26]:
# Import Library yang dibutuhkan
import time
import numpy as np
import imclab
import BlynkLib

print("Semua library siap.")

Semua library siap.


In [27]:
# --- KONFIGURASI BLYNK ---
# Dapatkan token dari Blynk Console: https://blynk.cloud
# Buat Template baru -> Tambah Device -> Salin Auth Token
BLYNK_AUTH = "J96xqMGDIjXZQZdtG4aMXS_NMqZ-x28j"
BLYNK_SERVER = "blynk.cloud"

# --- PEMETAAN VIRTUAL PIN BLYNK ---
# Virtual Pin untuk MENERIMA data dari Blynk App (VPIN_READ)
VPIN_SETPOINT = 0   # V0  -> Slider/Input setpoint RPM
VPIN_KP       = 4   # V4  -> Slider/Input Kp
VPIN_KI       = 5   # V5  -> Slider/Input Ki
VPIN_KD       = 6   # V6  -> Slider/Input Kd

# Virtual Pin untuk MENGIRIM data ke Blynk App (VPIN_WRITE)
VPIN_RPM      = 1   # V1  -> Label/Gauge RPM aktual
VPIN_OUTPUT   = 2   # V2  -> Label/Gauge PID output (%)
VPIN_ERROR    = 3   # V3  -> Label/Gauge error kontrol

# --- VARIABEL PID GLOBAL ---
# Nilai awal yang bisa diubah via Blynk App
setpoint = 0.0
kp = 0.5
ki = 0.1
kd = 0.005  # Gunakan nilai D kecil dulu
prev_error = 0.0
integral = 0.0
last_time = 0

# Variabel untuk Data Logging
log_data = []

print(f"Blynk Auth Token: {BLYNK_AUTH[:8]}...")
print(f"Server: {BLYNK_SERVER}")
print("Virtual Pins: V0=Setpoint | V1=RPM | V2=Output | V3=Error | V4=Kp | V5=Ki | V6=Kd")

Blynk Auth Token: J96xqMGD...
Server: blynk.cloud
Virtual Pins: V0=Setpoint | V1=RPM | V2=Output | V3=Error | V4=Kp | V5=Ki | V6=Kd


In [28]:
# --- INISIALISASI BLYNK & HANDLER VIRTUAL PIN ---

# Pastikan koneksi lama ditutup jika restart kernel
try:
    blynk.disconnect()
except:
    pass

blynk = BlynkLib.Blynk(BLYNK_AUTH, server=BLYNK_SERVER)

# Handler: dipanggil saat Blynk App mengubah nilai Setpoint (V0)
@blynk.VIRTUAL_WRITE(VPIN_SETPOINT)
def v0_write_handler(value):
    global setpoint, integral
    try:
        new_sp = float(value[0])
        if setpoint != new_sp:
            setpoint = new_sp
            integral = 0.0  # Reset integral saat setpoint berubah (anti-windup)
            print(f"Setpoint Diperbarui: {setpoint} RPM")
    except (ValueError, IndexError):
        pass

# Handler: dipanggil saat Blynk App mengubah nilai Kp (V4)
@blynk.VIRTUAL_WRITE(VPIN_KP)
def v4_write_handler(value):
    global kp
    try:
        kp = float(value[0])
        print(f"Koefisien Kp Diperbarui: {kp}")
    except (ValueError, IndexError):
        pass

# Handler: dipanggil saat Blynk App mengubah nilai Ki (V5)
@blynk.VIRTUAL_WRITE(VPIN_KI)
def v5_write_handler(value):
    global ki
    try:
        ki = float(value[0])
        print(f"Koefisien Ki Diperbarui: {ki}")
    except (ValueError, IndexError):
        pass

# Handler: dipanggil saat Blynk App mengubah nilai Kd (V6)
@blynk.VIRTUAL_WRITE(VPIN_KD)
def v6_write_handler(value):
    global kd
    try:
        kd = float(value[0])
        print(f"Koefisien Kd Diperbarui: {kd}")
    except (ValueError, IndexError):
        pass

print("Handler Virtual Pin Blynk sudah terdaftar.")
print("Mencoba koneksi ke Blynk...")

Handler Virtual Pin Blynk sudah terdaftar.
Mencoba koneksi ke Blynk...


In [29]:
# --- KONEKSI IMCLAB ---
try:
    lab = imclab.iMCLab()
    print("iMCLab/ESP32 Berhasil Terhubung!")
    time.sleep(1)
except Exception as e:
    print(f"GAGAL: {e}")
    lab = None

Opening connection
iMCLab connected via Arduino on port /dev/cu.usbserial-0001
iMCLab/ESP32 Berhasil Terhubung!


In [30]:
# --- MAIN PID LOOP ---
global prev_error, integral, last_time, log_data

# Interval update data ke Blynk App (detik)
BLYNK_UPDATE_INTERVAL = 1

if lab is None:
    print("Koneksi iMCLab tidak ditemukan.")
else:
    print(f"--- PID Loop Dimulai (Update Blynk setiap {BLYNK_UPDATE_INTERVAL}s) ---")

    start_time = time.time()
    last_time = start_time
    last_blynk_time = start_time

    try:
        while True:
            # Jalankan event loop Blynk (non-blocking)
            blynk.run()

            current_time = time.time()
            dt = current_time - last_time

            # --- 1. LOOP PID (CEPAT: 10Hz) ---
            if dt >= 0.1:

                # Baca RPM dari ESP32
                current_rpm = lab.RPM

                # Hitung PID
                error = setpoint - current_rpm

                P = kp * error

                integral += error * dt
                integral = max(-100, min(100, integral))  # Anti-windup
                I = ki * integral

                D = kd * (error - prev_error) / dt if dt > 0 else 0

                output_signal = P + I + D
                output_signal = max(0.0, min(100.0, output_signal))

                # Kirim ke Motor
                lab.op(output_signal)

                # Update variabel
                prev_error = error
                last_time = current_time

                # Logging Data
                log_data.append([current_time - start_time, output_signal, current_rpm, setpoint])

                # --- 2. LOOP BLYNK (LAMBAT: 1 Detik) ---
                if (current_time - last_blynk_time) > BLYNK_UPDATE_INTERVAL:

                    # Kirim data telemetri ke Blynk App
                    blynk.virtual_write(VPIN_RPM,    round(current_rpm, 0))
                    blynk.virtual_write(VPIN_OUTPUT, round(output_signal, 1))
                    blynk.virtual_write(VPIN_ERROR,  round(error, 1))
                    blynk.virtual_write(VPIN_SETPOINT, setpoint)

                    last_blynk_time = current_time

                    print(f"T: {current_time-start_time:.1f} | SP: {setpoint:.0f} | RPM: {current_rpm:.0f} | Out: {output_signal:.1f}%")

            time.sleep(0.005)

    except KeyboardInterrupt:
        print("\nPID Loop Dihentikan.")
    except Exception as e:
        print(f"\nError: {e}")

    finally:
        lab.op(0)
        print("Motor dimatikan.")

--- PID Loop Dimulai (Update Blynk setiap 1s) ---
T: 1.1 | SP: 0 | RPM: 0 | Out: 0.0%
T: 2.2 | SP: 0 | RPM: 0 | Out: 0.0%
T: 3.2 | SP: 0 | RPM: 0 | Out: 0.0%
T: 4.3 | SP: 0 | RPM: 0 | Out: 0.0%
T: 5.4 | SP: 0 | RPM: 0 | Out: 0.0%
T: 6.5 | SP: 0 | RPM: 0 | Out: 0.0%
Setpoint Diperbarui: 2108.0 RPM
T: 7.6 | SP: 2108 | RPM: 0 | Out: 100.0%
T: 8.6 | SP: 2108 | RPM: 6417 | Out: 0.0%
T: 9.7 | SP: 2108 | RPM: 2628 | Out: 0.0%
T: 10.8 | SP: 2108 | RPM: 0 | Out: 100.0%
T: 11.9 | SP: 2108 | RPM: 6389 | Out: 0.0%
T: 12.9 | SP: 2108 | RPM: 2651 | Out: 0.0%
T: 14.0 | SP: 2108 | RPM: 0 | Out: 100.0%
T: 15.1 | SP: 2108 | RPM: 6531 | Out: 0.0%
T: 16.2 | SP: 2108 | RPM: 2753 | Out: 0.0%
T: 17.3 | SP: 2108 | RPM: 0 | Out: 100.0%
T: 18.4 | SP: 2108 | RPM: 6514 | Out: 0.0%
Koefisien Kp Diperbarui: 1.0
T: 19.4 | SP: 2108 | RPM: 2806 | Out: 0.0%
T: 20.5 | SP: 2108 | RPM: 0 | Out: 100.0%
Koefisien Ki Diperbarui: 0.8
T: 21.6 | SP: 2108 | RPM: 6408 | Out: 0.0%
Koefisien Kp Diperbarui: 0.4
T: 22.7 | SP: 2108 | 